# Symmetry Phase 5 - Helper-Based Ablations

This notebook demonstrates collective-effect reconstruction:
1. ablate target-target couplings,
2. reconstruct target geometry using helpers,
3. compare top-coupling vs random helper strategies.

In [ ]:
import numpy as np

from boardGPT.analysis import (
    ablate_target_block,
    build_exponential_kernel_matrix,
    helper_reconstruction_curve,
    lattice_1d_concept,
    pairwise_distances,
    reconstruct_target_with_helpers,
    select_helpers_by_coupling,
)

## 1) Build a synthetic kernel matrix

In [ ]:
concept = lattice_1d_concept(num_tokens=64, periodic=True, period=2.0, name='ring64')
dist = pairwise_distances(concept)
matrix = build_exponential_kernel_matrix(distances=dist, amplitude=1.0, sigma=0.22, offset=0.05)
target = np.arange(12)

ablated = ablate_target_block(matrix, target_indices=target, value=0.0)
print('matrix shape:', matrix.shape)

## 2) Single reconstruction with top helpers

In [ ]:
helpers = select_helpers_by_coupling(ablated, target_indices=target, top_k=16)
coords_t, gram_t, used = reconstruct_target_with_helpers(
    matrix=ablated,
    target_indices=target,
    helper_indices=helpers,
    rank=12,
    mode='absolute',
)
print('target coords shape:', coords_t.shape)
print('used indices count:', used.shape[0])

## 3) Reconstruction curve vs helper size

In [ ]:
sizes = [0, 2, 4, 8, 12, 16, 24]
curve_top = helper_reconstruction_curve(
    original_matrix=matrix,
    ablated_matrix=ablated,
    target_indices=target,
    helper_sizes=sizes,
    strategy='top_coupling',
    num_trials=10,
    rank=12,
    mode='absolute',
    seed=0,
)
curve_rand = helper_reconstruction_curve(
    original_matrix=matrix,
    ablated_matrix=ablated,
    target_indices=target,
    helper_sizes=sizes,
    strategy='random',
    num_trials=10,
    rank=12,
    mode='absolute',
    seed=0,
)

print('sizes:', curve_top.helper_sizes.tolist())
print('top-coupling procrustes mean:', curve_top.procrustes_mean.tolist())
print('random      procrustes mean:', curve_rand.procrustes_mean.tolist())

You can plot `curve_top` and `curve_rand` with matplotlib for publication-style figures.